In [2]:
from openai import OpenAI
import json
import os
import re
from datetime import date, timedelta

import requests
from dotenv import load_dotenv, find_dotenv
from pydantic import BaseModel, Field

_ = load_dotenv(find_dotenv()) # read local .env file

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

client = OpenAI()

/Users/yasaman.nj/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [3]:
SYSTEM_PROMPT = """You are a friendly, practical AI travel advisor.
Help the user plan a personalized, realistic trip through conversation.
Before generating an itinerary, make sure you know all three essentials:
(1) destination,
(2) trip duration (number of days or dates), and
(3) at least one travel interest (for example food, museums, hiking, beaches,
nightlife, or shopping).

Ask one concise, natural follow-up covering only the missing essentials. Do not
ask for optional preferences if the user has not volunteered them; use sensible
assumptions instead.

If the destination, dates, duration, or other details conflict or are materially
ambiguous, ask the user to clarify before planning. Do not silently choose between
contradictory details. Treat an explicit "surprise me" destination as permission
to suggest a destination and proceed, while briefly stating your choice.

Once the essentials are clear and there is no unresolved contradiction, provide
the complete plan without asking for confirmation. Include a memorable trip title,
a short summary, a day-by-day itinerary with a sensible pace, practical travel
tips, and an estimated per-person budget with currency and a clear breakdown.
State assumptions, especially when dates, departure city, or budget are unknown.

Use the weather and exchange-rate function tools when current weather or currency
conversion is relevant to the user's request or itinerary. Weather forecasts are
only meaningful within the available forecast period; beyond that, describe
seasonal expectations instead. Label retrieved data with its date, and treat
exchange rates as approximate mid-market rates, not guaranteed card or bank rates.
Use web search for current public-transport options, operator information, or
service changes. Prefer official sources. Do not claim live availability or
opening hours. Include clickable source URLs for retrieved facts and recommend
checking details before booking.

Respect any accessibility, dietary, safety, or budget constraints the user mentions.
Be transparent when a specific recommendation or cost is uncertain.
"""

Token totals are accumulated from each API response's usage data. The cost is an estimate using the Standard `gpt-6-luna` rates listed in the [OpenAI API pricing guide](https://developers.openai.com/api/docs/pricing): $0.10 per 1M input tokens, $0.01 per 1M cached input tokens, and $0.50 per 1M output tokens. Update the rates if you change the model or pricing tier. Separate built-in web-search charges are not included.

In [4]:
MODEL = "gpt-6-luna"

PRICING_USD_PER_MILLION = {
    "gpt-6-luna": {"input": 0.10, "cached_input": 0.01, "output": 0.50}
}
history = [{"role": "system", "content": SYSTEM_PROMPT}]

conversation_usage = {
    "input_tokens": 0,
    "cached_input_tokens": 0,
    "output_tokens": 0,
    "estimated_cost_usd": 0.0,
}

class Structure(BaseModel):
    title: str = Field(description="A title for the trip")
    summary: str = Field(description="Trip summary")
    itinerary: str = Field(description="Day-by-day itinerary")
    practical_tips: str = Field(description="Practical travel tips")
    budget: str = Field(description="Estimated budget")


TOOLS = [
    {
        "type": "function",
        "name": "get_weather_forecast",
        "description": "Get a daily weather forecast for a location and dates within the next 16 days.",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {"type": "string", "description": "City and country"},
                "start_date": {"type": "string", "description": "First forecast date in YYYY-MM-DD format"},
                "days": {"type": "integer", "minimum": 1, "maximum": 16},
            },
            "required": ["location", "start_date", "days"],
            "additionalProperties": False,
        },
        "strict": True,
    },
    {
        "type": "function",
        "name": "get_exchange_rate",
        "description": "Get a recent exchange rate and convert an amount between two currencies.",
        "parameters": {
            "type": "object",
            "properties": {
                "base_currency": {"type": "string", "description": "Three-letter currency code"},
                "quote_currency": {"type": "string", "description": "Three-letter currency code"},
                "amount": {"type": "number", "description": "Amount in the base currency"},
            },
            "required": ["base_currency", "quote_currency", "amount"],
            "additionalProperties": False,
        },
        "strict": True,
    },
    {"type": "web_search"},
]


def get_weather_forecast(location, start_date, days):
    forecast_start = date.fromisoformat(start_date)
    forecast_end = forecast_start + timedelta(days=days - 1)
    today = date.today()
    if days < 1 or days > 16:
        return {"error": "Choose between 1 and 16 forecast days."}
    if forecast_start < today or forecast_end > today + timedelta(days=15):
        return {"error": "A forecast is only available for dates within the next 16 days. Use seasonal expectations for later trips."}

    geocoding_response = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": location, "count": 1, "language": "en", "format": "json"},
        timeout=20,
    )
    geocoding_response.raise_for_status()
    locations = geocoding_response.json().get("results", [])
    if not locations:
        return {"error": f"Could not find a location matching {location!r}."}

    place = locations[0]
    forecast_response = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": place["latitude"],
            "longitude": place["longitude"],
            "daily": "weather_code,temperature_2m_max,temperature_2m_min,precipitation_probability_max",
            "timezone": "auto",
            "start_date": forecast_start.isoformat(),
            "end_date": forecast_end.isoformat(),
        },
        timeout=20,
    )
    forecast_response.raise_for_status()
    forecast_data = forecast_response.json()
    return {
        "location": f"{place['name']}, {place.get('country', '')}",
        "timezone": forecast_data.get("timezone"),
        "daily_forecast": forecast_data.get("daily", {}),
        "retrieved_on": today.isoformat(),
        "sources": [
            {"title": "Open-Meteo geocoding", "url": geocoding_response.url},
            {"title": "Open-Meteo forecast", "url": forecast_response.url},
        ],
    }


def get_exchange_rate(base_currency, quote_currency, amount):
    base_currency = base_currency.upper()
    quote_currency = quote_currency.upper()
    if (
        not re.fullmatch(r"[A-Z]{3}", base_currency)
        or not re.fullmatch(r"[A-Z]{3}", quote_currency)
    ):
        return {"error": "Currency codes must each contain exactly three letters."}

    rate_response = requests.get(
        f"https://api.frankfurter.dev/v2/rate/{base_currency}/{quote_currency}",
        timeout=20,
    )
    rate_response.raise_for_status()
    rate_data = rate_response.json()
    return {
        "base_currency": base_currency,
        "quote_currency": quote_currency,
        "rate": rate_data["rate"],
        "rate_date": rate_data["date"],
        "amount": amount,
        "converted_amount": round(amount * rate_data["rate"], 2),
        "sources": [{"title": "Frankfurter exchange rates", "url": rate_response.url}],
    }


def execute_function_tool(name, arguments):
    try:
        if name == "get_weather_forecast":
            return get_weather_forecast(**arguments)
        if name == "get_exchange_rate":
            return get_exchange_rate(**arguments)
        return {"error": f"Unknown function tool: {name}"}
    except (requests.RequestException, KeyError, ValueError) as error:
        return {"error": f"Tool {name} could not retrieve data: {error}"}


def extract_web_sources(response):
    sources = []
    seen_urls = set()
    for output_item in response.output:
        if getattr(output_item, "type", None) != "message":
            continue
        for content in getattr(output_item, "content", []):
            for annotation in getattr(content, "annotations", []) or []:
                if getattr(annotation, "type", None) == "url_citation":
                    url = getattr(annotation, "url", None)
                    if url and url not in seen_urls:
                        sources.append((getattr(annotation, "title", None) or url, url))
                        seen_urls.add(url)
    return sources


def format_as_markdown(trip: Structure, sources=()):
    markdown = f"""# {trip.title}

## Summary
{trip.summary}

## Day-by-day itinerary
{trip.itinerary}

## Practical tips
{trip.practical_tips}

## Estimated budget
{trip.budget}"""
    if sources:
        source_links = "\n".join(f"- [{title}]({url})" for title, url in sources)
        markdown += f"\n\n## Sources\n{source_links}"
    return markdown


def send_request(input):
    return client.responses.parse(
        model=MODEL,
        input=input,
        text_format=Structure,
        tools=TOOLS,
    )


def chat(user_text):
    input_items = list(history)
    input_items.append({"role": "user", "content": user_text})
    tool_sources = []
    total_input_tokens = 0
    total_cached_input_tokens = 0
    total_output_tokens = 0

    for tool_round in range(6):
        response = send_request(input_items)
        usage = response.usage
        total_input_tokens += usage.input_tokens
        total_cached_input_tokens += getattr(
            getattr(usage, "input_tokens_details", None), "cached_tokens", 0
        ) or 0
        total_output_tokens += usage.output_tokens

        function_calls = [
            item for item in response.output
            if getattr(item, "type", None) == "function_call"
        ]
        if not function_calls:
            break
        if tool_round == 5:
            raise RuntimeError("Tool calling exceeded the maximum of six rounds.")

        input_items.extend(response.output)
        for function_call in function_calls:
            function_arguments = json.loads(function_call.arguments)
            tool_result = execute_function_tool(function_call.name, function_arguments)
            for source in tool_result.get("sources", []):
                tool_sources.append((source["title"], source["url"]))
            input_items.append({
                "type": "function_call_output",
                "call_id": function_call.call_id,
                "output": json.dumps(tool_result),
            })

    answer = response.output_text
    history.append({"role": "user", "content": user_text})
    history.append({"role": "assistant", "content": answer})

    conversation_usage["input_tokens"] += total_input_tokens
    conversation_usage["cached_input_tokens"] += total_cached_input_tokens
    conversation_usage["output_tokens"] += total_output_tokens

    rates = PRICING_USD_PER_MILLION[MODEL]
    uncached_input_tokens = total_input_tokens - total_cached_input_tokens
    conversation_usage["estimated_cost_usd"] += (
        uncached_input_tokens * rates["input"]
        + total_cached_input_tokens * rates["cached_input"]
        + total_output_tokens * rates["output"]
    ) / 1_000_000

    print(
        f"Conversation totals: {conversation_usage['input_tokens']:,} input tokens, "
        f"{conversation_usage['output_tokens']:,} output tokens; "
        f"estimated model cost ${conversation_usage['estimated_cost_usd']:.6f} USD"
    )
    sources = tool_sources + extract_web_sources(response)
    return format_as_markdown(response.output_parsed, sources)

In [5]:
from IPython.display import Markdown, display

trip_start = (date.today() + timedelta(days=2)).isoformat()
display(Markdown(chat(
    f"I want to visit Barcelona for 3 days starting {trip_start}. "
    "I love food and architecture. Include the weather forecast and convert "
    "500 USD to EUR."
)))
display(Markdown(chat(
    "What public transport options should I consider in Barcelona? Search for "
    "current official information."
)))

Conversation totals: 27,571 input tokens, 2,682 output tokens; estimated model cost $0.003637 USD


# Barcelona in Three Acts: Gaudí, Gothic Streets & Catalan Flavors

## Summary
Three days in Barcelona, Sunday, October 18 through Tuesday, October 20, 2026, pairing landmark architecture with market tastes and Catalan meals. The forecast looks wet for the first two days, so the plan prioritizes indoor visits early and a neighborhood walk on Tuesday. Assumptions: two nights in Barcelona, one traveler, and no flights included.

## Day-by-day itinerary
Day 1 — Sunday, October 18: Gaudí and a taste of the markets. Start with a timed visit to the Sagrada Família; book ahead through the [official ticket page](https://sagradafamilia.org/en/tickets/?lang=en). Have lunch around the basilica, then visit the Hospital de Sant Pau if you want another major architectural interior. The city’s events listing shows Mercat de Mercats, a food-market fair at Plaça de les Glòries, through October 18; consider it as a late-afternoon food stop if weather and energy allow. For dinner, try Catalan small plates such as pa amb tomàquet, seasonal vegetables, and seafood.  

Day 2 — Monday, October 19: Modernisme on Passeig de Gràcia. Visit La Pedrera (Casa Milà) in the morning, then take a relaxed lunch nearby. In the afternoon, tour Casa Batlló and admire the Modernist façades along Passeig de Gràcia. Both sites are indoors, making this your best rainy-day pairing. For dinner, choose a neighborhood restaurant in Eixample and try a rice or seafood dish. Check ticket times and details with the [official La Pedrera site](https://www.lapedrera.com/en/) and [Casa Batlló](https://www.casabatllo.es/en/).  

Day 3 — Tuesday, October 20: Gothic Quarter, El Born, and a final market bite. Take a slow walking loop through the Gothic Quarter and El Born, with coffee and a pastry along the way. Visit Santa Maria del Mar, then browse the Santa Caterina Market for lunch or snacks. If you prefer another architectural interior, swap some wandering time for a visit to the Palau de la Música Catalana. Leave the afternoon flexible for a final tapas meal and any favorite streets you want to revisit.

## Practical tips
Weather forecast retrieved October 16, 2026: October 18 is forecast around 16–19°C with an 80% chance of precipitation and possible thunderstorms; October 19 around 15–17°C with a 76% chance of rain; and October 20 around 14–22°C with a 58% chance of precipitation and mostly cloudy conditions. These are forecasts, not guarantees. Bring a compact umbrella and water-resistant shoes, and check again before you go. [Forecast source](https://api.open-meteo.com/v1/forecast?latitude=41.38879&longitude=2.15899&daily=weather_code%2Ctemperature_2m_max%2Ctemperature_2m_min%2Cprecipitation_probability_max&timezone=auto&start_date=2026-10-18&end_date=2026-10-20).  

For getting around, Barcelona’s TMB says the Hola Barcelona Travel Card provides unlimited rides for 24–120 hours and includes metro journeys to and from the airport; compare it with pay-as-you-go based on how often you’ll ride. Details: [TMB travel-card information](https://static-web.tmb.cat/en/barcelona-fares-metro-bus/tickets-visit-barcelona/barcelona-travel-card-hola-bcn). Book timed-entry architecture visits in advance, and recheck transport, event, and attraction details before booking; availability and schedules can change.  

Currency conversion retrieved October 16, 2026: **$500 USD ≈ €446.28 EUR** at an approximate mid-market rate of 1 USD = €0.89256. Your bank or card provider may use a different rate or charge fees. [Rate source](https://api.frankfurter.dev/v2/rate/USD/EUR).

## Estimated budget
Estimated per person for 3 days / 2 nights, excluding flights: **€595–€975** (roughly **$666–$1,092** using the October 16, 2026 rate). Breakdown: lodging €300–€500; meals and market tastings €150–€240; architecture admissions €80–€120; local transport €25–€35; coffee, snacks, and small extras €40–€80. These are planning estimates, not live prices; accommodation choice and ticket selections will shift the total.

## Sources
- [Open-Meteo geocoding](https://geocoding-api.open-meteo.com/v1/search?name=Barcelona%2C+Spain&count=1&language=en&format=json)
- [Open-Meteo forecast](https://api.open-meteo.com/v1/forecast?latitude=41.38879&longitude=2.15899&daily=weather_code%2Ctemperature_2m_max%2Ctemperature_2m_min%2Cprecipitation_probability_max&timezone=auto&start_date=2026-10-18&end_date=2026-10-20)
- [Frankfurter exchange rates](https://api.frankfurter.dev/v2/rate/USD/EUR)

KeyboardInterrupt: 

In [ ]:
import time

start = time.perf_counter()
display(Markdown(chat(
    f"I want to visit Barcelona for 3 days starting {trip_start}. "
    "I love food and architecture. Include the weather forecast and convert "
    "500 USD to EUR."
)))
print(f"Trip request took {time.perf_counter() - start:.1f}s")

start = time.perf_counter()
display(Markdown(chat(
    "What public transport options should I consider in Barcelona? Search for "
    "current official information."
)))
print(f"Transport request took {time.perf_counter() - start:.1f}s")

Conversation totals: 60,921 input tokens, 4,876 output tokens; estimated model cost $0.006962 USD


# Barcelona: Gaudí, Gothic Lanes & Catalan Flavors

## Summary
A three-day food-and-architecture escape, with landmark visits balanced against relaxed meals and rainy-day alternatives. Assumptions: one adult, two nights in Barcelona, no flights included, and a moderate-comfort budget. The forecast retrieved October 15, 2026, suggests thunderstorms on October 17–18, so keep outdoor plans flexible.

## Day-by-day itinerary
**Day 1 — Saturday, October 17: Sagrada Família & Eixample**
Forecast: high 24.5°C, low 16.1°C; thunderstorms indicated, with an 85% maximum precipitation probability. Visit the Sagrada Família in the morning, then have lunch nearby. Spend the afternoon exploring Passeig de Gràcia’s Modernisme architecture, with indoor visits to Casa Batlló or La Pedrera as weather-friendly options. Finish with tapas or Catalan cuisine in Eixample.

**Day 2 — Sunday, October 18: Gothic Quarter & El Born**
Forecast: high 19.2°C, low 14.3°C; thunderstorms indicated, with an 88% maximum precipitation probability. Explore the Gothic Quarter and El Born at an easy pace, taking shelter in cafés or indoor cultural stops if showers arrive. Have lunch around El Born or Santa Caterina, then enjoy a relaxed dinner featuring Catalan specialties or seafood.

**Day 3 — Monday, October 19: Park Güell & Gràcia**
Forecast: high 18.0°C, low 14.0°C; partly cloudy, with a 61% maximum precipitation probability. If conditions allow, visit Park Güell in the morning, then have lunch in Gràcia. If it is wet, swap the park for an indoor museum or architectural visit. Leave time for a final coffee, pastry, or market-food stop before departure.

## Practical tips
Forecast retrieved October 15, 2026: Oct 17 — 24.5°/16.1°C, thunderstorms indicated (85% max precipitation probability); Oct 18 — 19.2°/14.3°C, thunderstorms indicated (88%); Oct 19 — 18.0°/14.0°C, partly cloudy (61%). Forecasts can change; bring a compact umbrella and light rain layer, and check again before outdoor activities. Forecast provider: Open-Meteo.

For city travel, TMB lists the 2026 one-zone T-casual 10-journey ticket at €13. It is not valid for metro trips to or from the airport’s Aeroport T1/T2 stations, so check the airport ticket rules if using the metro from the airport. ([static-web.tmb.cat](https://static-web.tmb.cat/en/barcelona-fares-metro-bus/transport-ticket-fares?utm_source=openai)) Check current operator updates and attraction schedules before setting out; no live availability or opening hours are assumed.

The exchange-rate tool returned **500 USD ≈ €446.44**, using an approximate mid-market rate of 1 USD = €0.89287 on October 15, 2026. Your card or bank may apply a different rate or fees.

## Estimated budget
Estimated per person for 3 days and 2 nights, in EUR; flights excluded. Assumes a private room: lodging €320–€600; food and drinks €165–€270; architecture and museum entries €50–€90; local transport €25–€40; incidentals €30–€60. **Estimated total: €590–€1,060.** Actual costs depend on lodging, dining choices, and which attractions you visit. The converted €446.44 is a currency conversion, not an estimate of the full trip cost.

## Sources
- [Open-Meteo geocoding](https://geocoding-api.open-meteo.com/v1/search?name=Barcelona%2C+Spain&count=1&language=en&format=json)
- [Open-Meteo forecast](https://api.open-meteo.com/v1/forecast?latitude=41.38879&longitude=2.15899&daily=weather_code%2Ctemperature_2m_max%2Ctemperature_2m_min%2Cprecipitation_probability_max&timezone=auto&start_date=2026-10-17&end_date=2026-10-19)
- [Frankfurter exchange rates](https://api.frankfurter.dev/v2/rate/USD/EUR)
- [2026 Barcelona metro & bus fares: tickets & prices | Transports Metropolitans de Barcelona](https://static-web.tmb.cat/en/barcelona-fares-metro-bus/transport-ticket-fares?utm_source=openai)

Trip request took 22.6s
Conversation totals: 77,250 input tokens, 6,151 output tokens; estimated model cost $0.008587 USD


# Barcelona Public Transport: Best Options for a Short Visit

## Summary
For getting around central Barcelona, consider the metro and buses first, with trams, FGC, and Rodalies useful for specific routes or trips beyond the central sights. For the airport, compare the metro, train, and airport buses based on your terminal and destination.

## Day-by-day itinerary
**Around the city**
- **Metro:** A practical choice for longer cross-city trips. TMB’s network includes the airport line L9 Sud.
- **City buses:** Handy for routes where you’d rather travel above ground or there isn’t a convenient metro stop.
- **Trams:** Useful on parts of the tram network; check the route map for your destination.
- **FGC and Rodalies trains:** Consider these for destinations beyond the core sightseeing area or nearby towns. Check the operator’s service information for your travel date.

**Tickets to compare**
- **T-casual:** A single-person ticket for 10 journeys; the current 2026 one-zone fare is **€13**. It is not valid for metro trips to or from the airport’s Aeroport T1/T2 stations.
- **Hola Barcelona Travel Card:** Unlimited public transport for consecutive periods from 24 to 120 hours, with airport metro trips included. Compare its price with the rides you expect to take.
- **Single ticket:** **€2.90** for one journey on eligible services; it may cost more than a multi-ride ticket if you’ll travel often.
- **Airport metro ticket:** **€5.90** for a metro journey that includes the airport stations, if you are using a ticket that does not cover airport access.

For a three-day visit, a T-casual may suit you if 10 rides are enough and you arrange airport travel separately. If you want unlimited rides and airport metro access, compare the Hola Barcelona card’s current price with your plans. TMB’s official ticket information lists these 2026 fares and conditions. ([tmb.cat](https://www.tmb.cat/es/tarifas-metro-bus-barcelona/precios-titulos-transporte?utm_source=openai))

## Practical tips
**From Barcelona–El Prat Airport:**
- **Metro L9 Sud:** Serves both T1 and T2. Some tickets—including the T-casual—aren’t valid for journeys to or from the airport stations; check your ticket or buy the airport metro ticket.
- **R2 train:** The airport rail option; check current service details and terminal access before relying on it.
- **Airport buses:** The metropolitan authority lists Aerobús and conventional bus 46 among airport connections. Check the relevant operator’s route, fare, and schedule for your terminal and destination.

Official information: [TMB fares and tickets](https://www.tmb.cat/en/barcelona-fares-metro-bus), [TMB airport metro guidance](https://www.tmb.cat/en/visit-barcelona/public-transport/metro-airport), [AMB airport transport options](https://www.ambmobilitat.cat/Principales/Aeropuerto.aspx?idioma=1), and [Rodalies route information](https://rodalies.gencat.cat/en/sobre-rodalies/linies-i-estacions/servei_rodalia_barcelona/r2n/). Check official service updates and ticket rules shortly before travelling; schedules and conditions can change. ([tmb.cat](https://www.tmb.cat/es/tarifas-metro-bus-barcelona/precios-titulos-transporte?utm_source=openai))

## Estimated budget
For one traveler, the 2026 one-zone T-casual costs **€13 for 10 journeys**. A single ticket is **€2.90**, and a metro airport ticket is **€5.90** if required. Hola Barcelona may be better value if you expect frequent rides and want airport metro travel included; check the current price on TMB’s official fare page. Airport buses may have separate ticket conditions, so verify their price with the operator. ([tmb.cat](https://www.tmb.cat/es/tarifas-metro-bus-barcelona/precios-titulos-transporte?utm_source=openai))

## Sources
- [Tarifas metro y bus Barcelona 2026: precios y títulos | Transports Metropolitans de Barcelona](https://www.tmb.cat/es/tarifas-metro-bus-barcelona/precios-titulos-transporte?utm_source=openai)

Transport request took 15.7s
